# Infilling Score — frozen StarCoder2-3B, 128-token source-code adaptation
Raoof et al., ICLR 2025: [paper](https://proceedings.iclr.cc/paper_files/paper/2025/file/5709163243753c9c9ab7f4b4e5a8766d-Paper-Conference.pdf), [author implementation, pinned commit](https://github.com/neginraoof/infilling-score/tree/54ee252e378afffe09f4dbde5c2ca3adbf8ccfe0). The full method and source were inspected. This experiment implements one author-normalized Infilling Score from the Bayes-ratio derivation, five future tokens, bottom-20% arithmetic mean, higher = member. Appendix pseudocode differs from main equations/source; the README and research provenance document the precise interpretation and boundary indexing.

Frozen causal LM, no masked/bidirectional head, FIM prompt, reference model, training, score search or sweep. Analyze the first contiguous **128 baseline-tokenizer input IDs** in every language. Native special tokens count toward the span; the first input ID is context, never an invented BOS. All predictable, non-special/nonempty-offset targets in the span are considered (usually 127), including boundary targets with shortened/no future windows exactly as the pinned author code. Record every effective count and short/invalid span.

All original 1600/200/200/1000 manifests are validated first. Smoke selects 80/20/20/30 with seed 42; full preserves all saved rows. Only calibration selects the threshold. Source/dependency bootstrap follows corrected ReCaLL v2 infrastructure. GPU inference runs manually on Kaggle; no original-paper benchmark equivalence claim.

In [ ]:
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
UPLOAD_SOURCE = ""  # optional mounted directory containing extracted package files
UPLOAD_ZIP = ""  # optional explicit infilling_kaggle_source.zip path
PROJECT_ROOT = WORK_ROOT / "infilling-code-source-v2"
BASELINE_RUN = "/kaggle/input/datasets/sayonsujitmondal/sersem-baseline-1000-results3"
PATH_OVERRIDES = dict(shadow="", validation="", calibration="", heldout="", scores="", config="")
ADAPTIVE_SCORES = ""  # optional; without it inference runs and adaptive metrics are omitted
MODE = "smoke"  # "smoke" or "full"
SMOKE_SIZES = dict(shadow=80, validation=20, calibration=20, heldout=30)
RUN_ID = f"infilling-code-{MODE}-v2"
OUT = WORK_ROOT / "infilling_outputs" / RUN_ID
RESUME_FROM = ""  # same MODE and scientific identity only
CACHE_REUSE_FROM = ""  # optional exact-compatible infilling_cache.sqlite, e.g. smoke -> full
MODEL_CACHE_DIR = ""  # empty inherits the saved SERSEM target cache
MODEL_LOCAL_FILES_ONLY = True
ALLOW_MODEL_DOWNLOAD = False  # set True only to allow pinned model preparation on Kaggle
DATASET_CACHE_DIR = WORK_ROOT / "hf_cache/datasets"
SOURCE_ACCESS = "streaming"  # "streaming" or "cached" (existing Arrow cache only)
SESSION_BUDGET_MINUTES = 480
DISK_RESERVE_GIB = 3.0
WORK_ROOT.mkdir(parents=True, exist_ok=True)
# Frozen scientific settings are read and hash-bound from reproducibility.json.


## 1. Find and verify the source package
Attach the Infilling source Dataset as either the generated ZIP or extracted files. Leave `UPLOAD_SOURCE` and `UPLOAD_ZIP` empty for unique automatic discovery, or set either one to its mounted path. If the configured path is missing, this cell lists matching mounted packages. It verifies the source manifest and every file before adding the source tree to Python imports. The project extracts into the versioned `infilling-code-source-v2` directory.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, zipfile
SOURCE_MANIFEST_SHA256 = 'e32bc6ff41416edb3b12d221898f5983219a7f22138cb31dac113b409170243c'

configured_dir = Path(UPLOAD_SOURCE).expanduser() if UPLOAD_SOURCE else None
configured_zip = Path(UPLOAD_ZIP).expanduser() if UPLOAD_ZIP else None
if configured_dir is not None and not configured_dir.exists():
    print("Configured UPLOAD_SOURCE is missing:", configured_dir)
    configured_dir = None
if configured_zip is not None and not configured_zip.is_file():
    print("Configured UPLOAD_ZIP is missing:", configured_zip)
    configured_zip = None
markers = list(INPUT_ROOT.rglob("infilling_code_attack/source_manifest.json"))
archives = list(INPUT_ROOT.rglob("infilling_kaggle_source.zip"))
matching_dirs = sorted({p.parent.parent for p in markers} | {p.parent for p in archives})
print("Matching Infilling source mounts:", [str(p) for p in matching_dirs])
if configured_dir is None and configured_zip is not None:
    configured_dir = configured_zip.parent
if configured_dir is None:
    if len(matching_dirs) != 1:
        raise FileNotFoundError("Set UPLOAD_SOURCE or UPLOAD_ZIP to exactly one Infilling source Dataset")
    configured_dir = matching_dirs[0]
UPLOAD_SOURCE = configured_dir
if configured_zip is None:
    candidate_zip = UPLOAD_SOURCE / "infilling_kaggle_source.zip"
    configured_zip = candidate_zip if candidate_zip.is_file() else None
UPLOAD_ZIP = configured_zip

if not (PROJECT_ROOT / "infilling_code_attack/source_manifest.json").is_file():
    if PROJECT_ROOT.exists():
        raise FileExistsError("Incomplete PROJECT_ROOT; use a fresh versioned project directory")
    extracted_manifest = UPLOAD_SOURCE / "infilling_code_attack/source_manifest.json"
    if extracted_manifest.is_file():
        if hashlib.sha256(extracted_manifest.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
            raise ValueError("Outdated extracted Infilling source package")
        source_manifest = json.loads(extracted_manifest.read_text())
        PROJECT_ROOT.mkdir(parents=True)
        for name in list(source_manifest) + ["infilling_code_attack/source_manifest.json"]:
            destination = (PROJECT_ROOT / name).resolve()
            if not destination.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError("Unsafe extracted source path")
            source_file = (UPLOAD_SOURCE / name).resolve()
            if not source_file.is_relative_to(UPLOAD_SOURCE.resolve()) or not source_file.is_file():
                raise FileNotFoundError("Missing or unsafe source file: " + name)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_file, destination)
    else:
        if UPLOAD_ZIP is None or not UPLOAD_ZIP.is_file():
            raise FileNotFoundError("No extracted source manifest or infilling_kaggle_source.zip at " + str(UPLOAD_SOURCE))
        with zipfile.ZipFile(UPLOAD_ZIP) as archive:
            for entry in archive.infolist():
                if not (PROJECT_ROOT / entry.filename).resolve().is_relative_to(PROJECT_ROOT.resolve()):
                    raise ValueError("Unsafe source archive path")
            zipped_manifest = archive.read("infilling_code_attack/source_manifest.json")
            if hashlib.sha256(zipped_manifest).hexdigest() != SOURCE_MANIFEST_SHA256:
                raise ValueError("Outdated Infilling source ZIP")
            allowed = set(json.loads(zipped_manifest)) | {"infilling_code_attack/source_manifest.json"}
            if set(archive.namelist()) != allowed:
                raise ValueError("Unexpected source archive files")
            archive.extractall(PROJECT_ROOT)
manifest_path = PROJECT_ROOT / "infilling_code_attack/source_manifest.json"
if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != SOURCE_MANIFEST_SHA256:
    raise ValueError("Source manifest mismatch; use a fresh PROJECT_ROOT and rebuilt source ZIP")
source_manifest = json.loads(manifest_path.read_text())
required_sources = {"shared/kaggle_setup.py", "shared/requirements-kaggle.txt",
    "infilling_code_attack/reproducibility.json", "infilling_code_attack/method.py",
    "infilling_code_attack/inference.py", "infilling_code_attack/evaluation.py"}
if not required_sources.issubset(source_manifest):
    raise ValueError("Source package is missing required files: " + str(required_sources-set(source_manifest)))
for name, expected in source_manifest.items():
    source_file = (PROJECT_ROOT / name).resolve()
    if not source_file.is_relative_to(PROJECT_ROOT.resolve()) or not source_file.is_file():
        raise FileNotFoundError("Missing or unsafe source file: " + name)
    if hashlib.sha256(source_file.read_bytes()).hexdigest() != expected:
        raise ValueError("Source hash mismatch: " + name)
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "SERSEM-Poisoned-Chalice-Competition-2026"
sys.path.insert(0, str(UPSTREAM / "code"))


In [ ]:
# Set environment policy before importing the shared installer or any ML package.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HOME"] = str(WORK_ROOT / "hf_cache")
os.environ["HF_DATASETS_CACHE"] = str(DATASET_CACHE_DIR)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
packages = ["libenchant-2-2", "hunspell-en-us"]
status = subprocess.run(["dpkg-query", "-W", "-f=${Status}\n", *packages], capture_output=True, text=True)
if status.returncode != 0 or status.stdout.splitlines().count("install ok installed") != len(packages):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *packages], check=True)
from shared.kaggle_setup import install_dependencies, validate_environment
install_report = install_dependencies(PROJECT_ROOT / "shared/requirements-kaggle.txt", WORK_ROOT)
# If an already-loaded dependency changed, setup requests one restart. A fresh
# kernel rerun sees satisfied pins and skips pip instead of repeating installs.
environment = validate_environment(PROJECT_ROOT / "shared/requirements-kaggle.txt", require_cuda=True)


In [ ]:
# Lightweight CPU checks only: paper formula, source/data/cache behavior and setup.
for test_file in ("test_infilling.py", "test_kaggle_setup.py"):
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", test_file, "-v"],
                   cwd=PROJECT_ROOT, check=True)


In [ ]:
# Validate the saved numerical/API environment before loading partition manifests.
from adaptive_token_attack.data import resolve_paths
try:
    paths = resolve_paths(BASELINE_RUN, PATH_OVERRIDES, INPUT_ROOT)
except FileNotFoundError:
    mounted_runs = sorted({p.parent for p in INPUT_ROOT.rglob("config.json")
        if (p.parent / "heldout_test_scores.parquet").is_file()})
    print("Matching mounted SERSEM result directories:", [str(p) for p in mounted_runs])
    raise
base_cfg = json.loads(Path(paths["config"]).read_text())
if base_cfg.get("versions") != environment["versions"]:
    raise ValueError("Restore the exact saved SERSEM numerical/API versions before Infilling execution")
print("Saved baseline versions match all", len(environment["versions"]), "pinned distributions.")


## 2. Validate saved partitions and comparison inputs
The configured saved baseline is used by default; path resolution reports missing files and ambiguous runs. It validates the original complete 1,600/200/200/1,000 manifests before deterministic smoke selection. Adaptive comparison is optional. If supplied or uniquely mounted, its exact held-out identities are checked before inference; no unrelated result is substituted.

In [ ]:
import copy, random
import numpy as np
import pandas as pd
from adaptive_token_attack.data import load_run, select_partitions, frame_fingerprint
from adaptive_token_attack.features import load_frozen_target
from infilling_code_attack.cache import Cache, fingerprint
from infilling_code_attack.data import scoring_identity, prepare_plans, fixed_configuration
from infilling_code_attack.inference import TorchScorer
from infilling_code_attack.runner import run_scores
from infilling_code_attack.evaluation import exact_match, evaluate
from infilling_code_attack import METHOD_NAME, METHOD_STATUS
from shared.resource_pipeline import Monitor, restore_run, file_hash
from shared.resource_data import SourceStore
from shared.sersem_baseline import write_json
fixed = fixed_configuration()
print(METHOD_NAME, METHOD_STATUS)
original, saved_scores, loaded_cfg = load_run(paths)
if loaded_cfg != base_cfg:
    raise ValueError("Baseline config changed between environment and manifest validation")
if SMOKE_SIZES != fixed["smoke_counts"]:
    raise ValueError("Smoke counts are frozen to 80/20/20/30")
partitions, selection = select_partitions(original, MODE, SMOKE_SIZES, seed=fixed["seed"])
if set(pd.concat(list(original.values())).language) != {"Go", "Java", "Python", "Ruby", "Rust"}:
    raise ValueError("Unexpected saved programming languages")
adaptive_scores = None
adaptive_path = None
if ADAPTIVE_SCORES:
    adaptive_path = Path(ADAPTIVE_SCORES).expanduser()
elif len(adaptive_candidates := sorted(INPUT_ROOT.rglob("heldout_adaptive_scores.parquet"))) == 1:
    adaptive_path = adaptive_candidates[0]
if adaptive_path is not None:
    print("Including optional adaptive comparison from:", adaptive_path)
    adaptive_scores = pd.read_parquet(adaptive_path)
    exact_match(partitions["heldout"], adaptive_scores, ["adaptive_score"])
else:
    print("No unique adaptive scores supplied; inference continues without adaptive comparison metrics.")
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in {".", ".."}:
    raise ValueError("RUN_ID must be a directory name")
if OUT.resolve().is_relative_to(INPUT_ROOT.resolve()) or any(p.resolve().is_relative_to(OUT.resolve()) for p in paths.values()):
    raise ValueError("OUT must be writable and separate from baseline inputs")
OUT.mkdir(parents=True, exist_ok=True)
if RESUME_FROM:
    restore_run(RESUME_FROM, OUT)
source_cfg = copy.deepcopy(base_cfg)
source_cfg["resources"].update(data_access=SOURCE_ACCESS, text_cache_gib=.1,
    session_budget_minutes=SESSION_BUDGET_MINUTES, disk_reserve_gib=DISK_RESERVE_GIB)
monitor = Monitor(source_cfg, OUT)
monitor.guard()
print("Selected:", {k:len(v) for k,v in partitions.items()})


## 3. Freeze the paper-derived scoring configuration
Load the source-pinned `reproducibility.json`; verify the saved manifest selection, tokenizer, model, precision and score settings before model loading. The fixed span is 128 IDs, future count 5, bottom fraction 0.2, higher-is-member, seed 42, and the documented author-code normalization and boundaries. No setting is selected from validation or held-out results.

In [ ]:
from shared.cache_setup import ensure_model_snapshot
from transformers import AutoConfig, AutoTokenizer
cache_dir = MODEL_CACHE_DIR or base_cfg["resources"]["model_cache_dir"]
ensure_model_snapshot(base_cfg["model_id"], base_cfg["model_revision"], cache_dir,
    ALLOW_MODEL_DOWNLOAD, DISK_RESERVE_GIB, WORK_ROOT, OUT / "model_cache_setup.json")
options = dict(revision=base_cfg["model_revision"], cache_dir=str(cache_dir), local_files_only=MODEL_LOCAL_FILES_ONLY)
tokenizer = AutoTokenizer.from_pretrained(base_cfg["model_id"], use_fast=True, **options)
model_config = AutoConfig.from_pretrained(base_cfg["model_id"], **options)
identity = scoring_identity(base_cfg, tokenizer, model_config, source_manifest)
experiment = dict(schema="infilling-code-experiment-v1", method_name=METHOD_NAME, method_status=METHOD_STATUS,
    statistic="pinned author normalized ratio", mode=MODE, selection=selection, scoring=identity,
    source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    inputs={k:hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
    adaptive_scores_sha256=(hashlib.sha256(adaptive_path.read_bytes()).hexdigest() if adaptive_path else None))
if (OUT / "config.json").exists() and json.loads((OUT / "config.json").read_text()) != experiment:
    raise ValueError("Execution identity changed; choose a new RUN_ID")
write_json(OUT / "config.json", experiment)
write_json(OUT / "reproducibility.json", fixed)
write_json(OUT / "research_provenance.json", json.loads((PROJECT_ROOT / "infilling_code_attack/research_provenance.json").read_text()))
write_json(OUT / "source_version_hashes.json", dict(source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    sources=source_manifest, versions=environment, scoring_sha256=fingerprint(identity)))
write_json(OUT / "input_paths.json", dict(**{k:str(v) for k,v in paths.items()},
    adaptive_scores=str(adaptive_path) if adaptive_path else None))
for role, frame in partitions.items():
    frame.to_parquet(OUT / (role + "_manifest.parquet"), index=False)
write_json(OUT / "run_manifest.json", dict(status="preparing", method_name=METHOD_NAME,
    statistic="pinned author normalized ratio", mode=MODE, selection=selection,
    scoring_sha256=fingerprint(identity)))
cache = Cache(OUT / "infilling_cache.sqlite")
if CACHE_REUSE_FROM:
    monitor.guard(Path(CACHE_REUSE_FROM).stat().st_size)
    cache.reuse_from(CACHE_REUSE_FROM)
source_store = SourceStore(source_cfg, OUT, monitor)
plans = prepare_plans(partitions, tokenizer, identity, cache, source_store, monitor)
print("Prepared plans:", len(plans))


## 4. Frozen StarCoder2 inference with resumable checkpoints
The model remains frozen; inference uses `torch.inference_mode()` and the T4-compatible precision fixed in the source configuration. SQLite stores only required sufficient statistics and per-query checkpoints. Session/disk guards run before work. Resume only with the same run identity, or import compatible cache records into a distinct smoke/full run.

In [ ]:
def scorer_factory():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Missing query statistics require a Kaggle CUDA GPU")
    monitor.guard()
    random.seed(fixed["seed"]); np.random.seed(fixed["seed"])
    torch.manual_seed(fixed["seed"]); torch.cuda.manual_seed_all(fixed["seed"])
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model, loaded_tokenizer = load_frozen_target(base_cfg, cache_dir, MODEL_LOCAL_FILES_ONLY)
    if hashlib.sha256(loaded_tokenizer.backend_tokenizer.to_str().encode()).hexdigest() != identity["tokenizer_sha256"]:
        raise ValueError("Inference tokenizer differs from prepared span plans")
    pad_id = loaded_tokenizer.pad_token_id
    if pad_id is None: pad_id = loaded_tokenizer.eos_token_id
    if pad_id is None: pad_id = 0
    return TorchScorer(model, pad_id, identity["head_chunk_rows"], identity["min_sigma"])
write_json(OUT / "run_manifest.json", dict(status="scoring", method_name=METHOD_NAME,
    method_status=METHOD_STATUS, mode=MODE, selection=selection,
    scoring_sha256=fingerprint(identity), source_manifest_sha256=SOURCE_MANIFEST_SHA256))
scored, timing = run_scores(partitions, plans, identity, cache, scorer_factory, monitor, OUT)
import gc, torch
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Processed:", len(scored), "invalid:", int(scored.status.eq("invalid").sum()))
print(json.dumps(timing, indent=2))


## 5. Calibrate, evaluate and compare exact identities
Only valid calibration rows set the deployed threshold. Held-out ROC thresholds remain diagnostics. Baseline static SERSEM, LUMIA and fusion are required and identity checked; adaptive scores are compared only when explicitly available and matched. Invalid Infilling rows remain in the score export, while all compared methods use the same explicitly valid held-out cohort.

In [ ]:
report = evaluate(scored, partitions, saved_scores, adaptive_scores, OUT, MODE, timing=timing)
monitor.guard()
cache.close()  # close/checkpoint SQLite before the results archive cell
write_json(OUT / "run_manifest.json", dict(status=report["status"], method_name=METHOD_NAME,
    method_status=METHOD_STATUS, mode=MODE, selection=selection,
    scoring_sha256=fingerprint(identity), source_manifest_sha256=SOURCE_MANIFEST_SHA256,
    author_commit=identity["author_commit"], input_hashes=experiment["inputs"],
    adaptive_scores_sha256=experiment["adaptive_scores_sha256"], selected_counts={k:len(v) for k,v in partitions.items()},
    coverage=report["coverage"], invalid_samples=report["invalid_samples"], timing=timing,
    span_summary=dict(short_spans=int(scored.short_span.sum()),
        fewer_than_128_eligible=int(scored.fewer_than_128_eligible_tokens.sum())),
    artifacts={p.name:file_hash(p) for p in OUT.iterdir() if p.is_file() and p.name in {
        "config.json", "metrics.json", "infilling_scores.parquet", "matched_comparison.parquet",
        "infilling_cache.sqlite", "source_version_hashes.json", "threshold.json", "timing.json",
        "reproducibility.json", "research_provenance.json"}}))
print(json.dumps(report, indent=2))


## 6. Validate and archive the completed results
This final step checks required outputs and SQLite integrity, confirms the cache writer has been closed, and creates a ZIP named from `RUN_ID`. It archives the complete `OUT` directory, including reproducibility metadata and resume cache.

In [ ]:
from infilling_code_attack.archive import create_results_archive
archive = create_results_archive(OUT, WORK_ROOT, RUN_ID)
assert archive.is_file() and archive.stat().st_size > 0
print("Saved results archive:", archive)


## Persist and resume
Save the entire `OUT` directory using Kaggle Save Version with outputs before ending a session. The final notebook cell also creates `WORK_ROOT / f"{RUN_ID}.zip"` after validating required outputs and the closed, checkpointed SQLite cache.

Same-run resume uses the same MODE/RUN_ID/scientific identity and `RESUME_FROM` set to saved OUT. Smoke-to-full uses a new full RUN_ID, empty RESUME_FROM, and optionally `CACHE_REUSE_FROM` pointing to the smoke cache; cache reuse imports only exact-compatible feature/query records. Adaptive scores are optional throughout. `reproducibility.json` remains fixed: seed 42, 128-token span, five future tokens, bottom-20% mean, original normalization, boundary policy and higher-is-member direction.

No StarCoder2 inference, Kaggle GPU verification, or real-data result generation was performed locally. Run on a Kaggle Tesla T4; see the package README for the pinned method interpretation and source provenance.